In [ ]:
from fastai.vision.all import *
from torch.nn import *
from itertools import permutations

set_num_threads(1)
torch.set_num_threads(1)


class net(dict):
    def __call__(s, _i, _p, dev="cuda:1", **a):
        r = []
        for _p in _p.split("\n"):
            f = s.get(_p[0], Dot)
            _p, _o = _p.partition("->")[::2]
            _o = _o or _i
            r.append(f(_i, _p[int(f is not Dot) :], _o, r, **a))
            _i = _o
        return Learner(
            DataLoaders(dat(**a, bs=16, it=9999, dev=dev), dat(**a, bs=1024, it=1, dev=dev)),
            Sequential(*r).to(dev),
            Los(),
            metrics=acc(10),
            cbs=ShowGraphCallback,
        )

    def reg(s, a):
        s[a.s] = a


net = net()


class dat:
    def __init__(s, e, l, bs, it, dev, **_):
        s.e = e
        s.l = l
        s.bs = bs
        s.it = it
        s.dev = dev

    def __len__(s):
        return s.it

    def __iter__(s):
        for _ in range(s.it):
            _ = torch.randint(s.e, [s.bs, s.l], device=s.dev)
            yield _.flatten(), _.sum(1) & (s.e - 1)


class Los(NLLLoss):
    def forward(s, y, t):
        return super().forward(y.log_softmax(2).mean(0), t)


def acc(n):
    def avg(y, t):
        nonlocal _
        t = (y.argmax(2) == t).float().mean(1)
        _ = torch.zeros(n, dtype=torch.long, device=y.device).put_(
            *torch.broadcast_tensors((t * (n - 1e-3)).long(), torch.tensor(1, device=y.device)), True
        ).float() * (100 / len(y))
        return t.mean() * 100

    yield avg
    for i, j in enumerate(np.linspace(0, 100, n, 0, dtype=int)):

        def _(y, t, i=i):
            return _[i]

        _.__name__ = str(j)
        yield _


class Dot(Module):
    def __init__(s, _i, _p, _o, _r, **a):
        super().__init__()
        s.w = ParameterList([Parameter(init.kaiming_uniform_(torch.empty([a[p] for p in p]), 5 ** 0.5)) for p in _p.split(",")])
        s.p = s.trans(f"{_i},{_p}->{_o}")

    @staticmethod
    def trans(p):
        return p.translate(str.maketrans(dict(zip(set(p) ^ {",", "-", ">"}, "abcdefg"))))

    def forward(s, x):
        return torch.einsum(s.p, x, *s.w)


@net.reg
class Get(Module):
    s = "#"

    def __init__(s, _i, _p, _o, _r, **a):
        super().__init__()
        s.w = Parameter(torch.randn([a[_p] for _p in _p[1:]]))
        s.i = tuple([slice(None)] * int(_p[0]))

    def forward(s, x):
        return s.w[s.i + (x,)]


@net.reg
class Bn(BatchNorm1d):
    s = "~"

    def __init__(s, _i, _p, _o, _r, **a):
        super().__init__(np.prod([a[_p] for _p in _p]), track_running_stats=0)
        s.f = sorted(range(len(_i)), key=lambda x: _i[x] in _p)
        s.b = [None] * len(_i)
        for _i, _p in enumerate(s.f):
            s.b[_p] = _i

    def forward(s, x):
        x = x.permute(s.f)
        return super().forward(x.reshape(-1, len(s.bias))).view_as(x).permute(s.b)


@net.reg
class Call(Module):
    s = "@"

    def __new__(s, _i, _p, _o, _r, **a):
        _p = eval(_p, None, a)
        if isinstance(_p, Module):
            return _p
        s = super().__new__(s)
        s.p = _p
        return s

    def __init__(s, *_, **__):
        super().__init__()

    def forward(s, x):
        return s.p(x)


@net.reg
class Add(Module):
    s = "+"

    def __init__(s, _i, _p, _o, _r, **a):
        super().__init__()
        s.w = Parameter(torch.zeros(s.trans([a[_i] if _i in _p else 1 for _i in _i])))

    @staticmethod
    def trans(a):
        while a[0] == 1:
            del a[0]
        return a

    def forward(s, x):
        return x + s.w


@net.reg
class View(Module):
    s = "-"

    def __init__(s, _i, _p, _o, _r, **a):
        super().__init__()
        s.s = [a[_o] if _o != "?" else -1 for _o in _o]

    def forward(s, x):
        return x.reshape(s.s)


@net.reg
class Rec(Sequential):
    s = "%"

    def __init__(s, _i, _p, _o, _r, **a):
        super().__init__(*_r[-int(_p[0]) :] * int(_p[1:] or "1"))


class MGLinear(Linear):
    "mn(gk), mK@gk, +mK, @Gg -> mnKG"

    def __init__(s, m, g, k, K, b):
        super().__init__(k, K, b)
        s.weight = Parameter(torch.randn(m, K, g, k))
        if b:
            s.bias = Parameter(torch.zeros(m, 1, K, 1))
        s.reset_parameters()
        s.g = torch.tensor(list(permutations(range(g))))

    def forward(s, x):
        x = (x @ s.weight[:, :, s.g].view(len(x), -1, x.size(2)).transpose(1, 2)).view(*x.shape[:2], -1, len(s.g))
        return x if s.bias is None else x + s.bias

In [ ]:
m = """#1mek->m?k
->m?K
@MGLinear(m,2,k,k,0)->m?kg
~mk
@Swish()
@lambda x:x.sum(3)->m?k
%53
mek->m?e
+me"""
m = net("?", m, m=1024, e=16, k=4, K=8, l=16, dev="cuda")
m.lr_find()

In [ ]:
m.fit_one_cycle(20, .001, 9, 99, .01)